# Chromatin Velocity Analysis Demo

This notebook demonstrates the chromatin velocity pipeline using co-accessibility propagation.

## Concept
Chromatin velocity adapts the RNA velocity framework to chromatin accessibility data:
- **"Spliced" analog**: Current accessibility of each peak
- **"Unspliced" analog**: Weighted sum of co-accessible peaks' accessibility (propagated signal)

This allows us to infer directionality in chromatin accessibility changes across development.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt
import seaborn as sns

# Add scripts directory to path
sys.path.append('../scripts')

# Import our chromatin velocity modules
from chromatin_velocity import ChromatinVelocity, compute_chromatin_velocity_pipeline
from chromatin_velocity_scvelo import ChromatinVelocityAnalysis, run_chromatin_velocity_analysis
from chromatin_velocity_viz import ChromatinVelocityVisualizer, create_publication_figure
from chromatin_velocity_validation import ChromatinVelocityValidator, run_comprehensive_validation

# Set up plotting
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")
sc.settings.verbosity = 3

## 1. Data Preparation

Load the required input data:
- Peak accessibility matrix (peaks × pseudobulks)
- Co-accessibility matrix from Cicero
- Peak UMAP coordinates and annotations

In [ ]:
# File paths - update these with your actual data paths
peaks_adata_path = "/path/to/peaks_by_pb_leiden_0.4_merged_annotated_filtered.h5ad"
coaccessibility_path = "/path/to/cicero_coaccessibility_matrix.csv"
output_dir = "./chromatin_velocity_results/"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Define developmental timepoint order
timepoint_order = ['0somites', '5somites', '10somites', '15somites', '20somites', '30somites']

print("Data preparation complete!")
print(f"Output directory: {output_dir}")
print(f"Timepoint order: {timepoint_order}")

## 2. Quick Start - Complete Pipeline

Run the entire chromatin velocity pipeline with a single function call.

In [ ]:
# Run complete chromatin velocity pipeline
cv = compute_chromatin_velocity_pipeline(
    peaks_adata_path=peaks_adata_path,
    coaccessibility_path=coaccessibility_path,
    output_path=f"{output_dir}/chromatin_velocity_results.h5ad",
    timepoint_order=timepoint_order,
    coaccess_threshold=0.1,  # Minimum co-accessibility score
    max_connections=100,     # Maximum connections per peak
    normalize=True           # Log-normalize accessibility
)

print("\nPipeline complete!")
print(f"Computed velocity for {len(cv.peak_names)} peaks")
print(f"Using {len(cv.pseudobulk_names)} pseudobulk samples")

## 3. Detailed Analysis with scVelo Integration

Perform detailed velocity analysis using the scVelo framework.

In [ ]:
# Run detailed scVelo analysis
cva = run_chromatin_velocity_analysis(
    chromatin_velocity=cv,
    mode='dynamical',        # Use dynamical mode for better velocity estimation
    n_neighbors=30,          # Number of neighbors for analysis
    min_confidence=0.75,     # Minimum confidence for velocity genes
    output_dir=output_dir
)

print("\nscVelo analysis complete!")
print(f"Analyzed {cva.adata.n_obs} peaks")
print(f"Using {cva.adata.n_vars} pseudobulk samples")

## 4. Visualization

Create comprehensive visualizations of chromatin velocity results.

In [ ]:
# Initialize visualizer
visualizer = ChromatinVelocityVisualizer(cva.adata, figsize=(10, 8))

# Create comprehensive summary figure
visualizer.create_velocity_summary_figure(
    basis='umap',
    save=f"{output_dir}/chromatin_velocity_summary.pdf"
)

In [ ]:
# Plot velocity embedding
cva.plot_velocity_embedding(
    basis='umap',
    color='velocity_confidence',
    save=f"{output_dir}/velocity_embedding.pdf",
    title='Chromatin Velocity on Peak UMAP'
)

In [ ]:
# Plot velocity stream
cva.plot_velocity_stream(
    basis='umap',
    color='leiden_coarse',
    save=f"{output_dir}/velocity_stream.pdf",
    title='Chromatin Velocity Stream'
)

In [ ]:
# Plot velocity by peak type
visualizer.plot_peak_type_velocity(
    peak_type_col='peak_type',
    save=f"{output_dir}/velocity_by_peak_type.pdf"
)

In [ ]:
# Plot temporal velocity trends
visualizer.plot_temporal_velocity_trends(
    timepoint_col='timepoint_order',
    celltype_col='celltype',
    save=f"{output_dir}/temporal_velocity_trends.pdf"
)

## 5. Biological Interpretation

Analyze the biological significance of chromatin velocity results.

In [ ]:
# Identify high-velocity pseudobulks (developmental transitions)
velocity_pseudobulks = cva.identify_velocity_genes(
    min_confidence=0.75,
    top_n=20
)

print("High-velocity pseudobulks (developmental transitions):")
for pb in velocity_pseudobulks[:10]:
    print(f"  - {pb}")

In [ ]:
# Analyze temporal patterns
temporal_results = cva.analyze_temporal_patterns(
    timepoint_col='timepoint_order',
    celltype_col='celltype'
)

if not temporal_results.empty:
    print("\nTemporal velocity patterns:")
    print(temporal_results.groupby('celltype')['velocity_magnitude'].agg(['mean', 'std']).round(3))
    
    # Save temporal results
    temporal_results.to_csv(f"{output_dir}/temporal_velocity_patterns.csv", index=False)

## 6. Validation

Validate chromatin velocity results using multiple approaches.

In [ ]:
# Run comprehensive validation
validator = run_comprehensive_validation(
    adata=cva.adata,
    coaccessibility_matrix=cv.coaccessibility_matrix,
    output_dir=f"{output_dir}/validation/"
)

print("\nValidation complete!")
print("Check validation_report.md for detailed results.")

In [ ]:
# Display validation summary
if 'temporal_consistency' in validator.validation_results:
    tc = validator.validation_results['temporal_consistency']
    print(f"\nTemporal Consistency:")
    print(f"  Mean correlation: {tc['mean_correlation']:.3f}")
    print(f"  Positive correlations: {tc['fraction_positive_correlation']:.1%}")
    print(f"  Significant correlations: {tc['fraction_significant']:.1%}")

if 'regulatory_coherence' in validator.validation_results:
    rc = validator.validation_results['regulatory_coherence']
    print(f"\nRegulatory Coherence:")
    print(f"  Clusters analyzed: {rc['n_clusters_analyzed']}")
    print(f"  Mean coherence: {rc['mean_coherence_score']:.3f}")

if 'pioneer_peaks' in validator.validation_results:
    pp = validator.validation_results['pioneer_peaks']
    print(f"\nPioneer Peaks:")
    print(f"  Identified: {pp['n_pioneer_peaks']} peaks")
    print(f"  Examples: {pp['pioneer_peak_names'][:5]}")

## 7. Advanced Analysis Examples

Demonstrate additional analysis capabilities.

In [ ]:
# Example: Focus on specific peak clusters
if 'leiden_coarse' in cva.adata.obs.columns:
    # Analyze velocity for neuronal peaks (example cluster)
    neuronal_cluster = '0'  # Replace with actual neuronal cluster ID
    neuronal_peaks = cva.adata.obs_names[cva.adata.obs['leiden_coarse'] == neuronal_cluster]
    
    print(f"\nNeuronal peak cluster analysis:")
    print(f"  Peaks in cluster {neuronal_cluster}: {len(neuronal_peaks)}")
    
    # Compute average velocity for this cluster
    neuronal_mask = cva.adata.obs['leiden_coarse'] == neuronal_cluster
    neuronal_velocity = cva.adata.layers['velocity'][neuronal_mask, :]
    mean_neuronal_velocity = np.mean(neuronal_velocity, axis=0)
    
    # Find timepoints with highest velocity
    if hasattr(cv, 'pseudobulk_metadata'):
        velocity_df = pd.DataFrame({
            'pseudobulk': cv.pseudobulk_names,
            'velocity': mean_neuronal_velocity
        })
        velocity_df = velocity_df.merge(cv.pseudobulk_metadata, on='pseudobulk')
        
        top_velocity = velocity_df.nlargest(5, 'velocity')
        print(f"  Top velocity timepoints for neuronal peaks:")
        for _, row in top_velocity.iterrows():
            print(f"    {row['pseudobulk']}: {row['velocity']:.3f}")

In [ ]:
# Example: Identify co-accessible peak modules with coherent velocity
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

# Sample subset of peaks for clustering
n_sample = min(1000, cva.adata.n_obs)
sample_indices = np.random.choice(cva.adata.n_obs, n_sample, replace=False)

# Get velocity vectors for sampled peaks
sample_velocity = cva.adata.layers['velocity'][sample_indices, :]

# Compute pairwise distances based on velocity similarity
velocity_distances = pdist(sample_velocity, metric='cosine')

# Hierarchical clustering
linkage_matrix = linkage(velocity_distances, method='ward')
clusters = fcluster(linkage_matrix, t=10, criterion='maxclust')

print(f"\nVelocity-based clustering:")
print(f"  Identified {len(np.unique(clusters))} velocity modules")
unique, counts = np.unique(clusters, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"    Module {cluster_id}: {count} peaks")

## 8. Save Results and Create Publication Figure

Save all results and create publication-ready figures.

In [ ]:
# Save final AnnData object with all velocity results
cva.adata.write_h5ad(f"{output_dir}/chromatin_velocity_final_results.h5ad")

# Create publication figure
create_publication_figure(
    adata=cva.adata,
    output_path=f"{output_dir}/chromatin_velocity_publication_figure.pdf",
    figure_size=(12, 8),
    dpi=300
)

print("\nResults saved!")
print(f"Main results: {output_dir}/chromatin_velocity_final_results.h5ad")
print(f"Publication figure: {output_dir}/chromatin_velocity_publication_figure.pdf")
print(f"Validation report: {output_dir}/validation/validation_report.md")

## Summary

This notebook demonstrated the complete chromatin velocity pipeline:

1. **Data Preparation**: Loading peak accessibility and co-accessibility data
2. **Velocity Computation**: Using co-accessibility propagation to compute "spliced" and "unspliced" analogs
3. **scVelo Integration**: Applying RNA velocity methodology to chromatin data
4. **Visualization**: Creating comprehensive plots of velocity patterns
5. **Biological Interpretation**: Identifying developmental transitions and regulatory modules
6. **Validation**: Ensuring biological relevance through multiple validation approaches

The chromatin velocity approach provides insights into:
- Temporal dynamics of chromatin accessibility
- Directional changes in regulatory landscapes
- Pioneer regulatory regions driving accessibility changes
- Coherent regulatory programs across development

This methodology opens new avenues for understanding chromatin dynamics during development and disease.